In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

#import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
 #   for filename in filenames:
  #      print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [8]:
import numpy as np
import cv2
import os

In [9]:
input_dir = '/kaggle/input/sahaay-human-dataset/Sahaay_Human_Dataset'
output_dir = '/kaggle/working/processed_images'

os.makedirs(output_dir, exist_ok=True)

In [10]:
import os
import json

# Suppose your training directory is:
train_dir = '/kaggle/input/sahaay-human-dataset/Sahaay_Human_Dataset'
class_names = sorted(os.listdir(train_dir))

# Save the class names to a JSON file
with open('class_names.json', 'w') as f:
    json.dump(class_names, f)


In [11]:
for subfolder in os.listdir(input_dir):
    subfolder_path = os.path.join(input_dir, subfolder)
    output_subfolder = os.path.join(output_dir, subfolder)
    os.makedirs(output_subfolder, exist_ok=True)
    for file in os.listdir(subfolder_path):
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            file_path = os.path.join(subfolder_path, file)
            img = cv2.imread(file_path)
            resized_img = cv2.resize(img, (224, 224))
            normalized_img = resized_img / 255.0
            processed_img = (normalized_img * 255).astype(np.uint8)
            output_path = os.path.join(output_subfolder, file)
            cv2.imwrite(output_path, processed_img)
    print("done") 

            

done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done


In [12]:
import os


dataset_dir = '/kaggle/working/processed_images'

avg = []
for class_name in os.listdir(dataset_dir):
    class_path = os.path.join(dataset_dir, class_name)
    if os.path.isdir(class_path):
        file_count = len([f for f in os.listdir(class_path) 
                          if os.path.isfile(os.path.join(class_path, f))])
        avg.append(file_count)
        print(f"Class '{class_name}' has {file_count} files.")


Class 'Basal Cell Carcinoma' has 5266 files.
Class 'SkinCancer' has 693 files.
Class 'Melanoma' has 4805 files.
Class 'DrugEruption' has 547 files.
Class 'Actinic keratosis' has 1600 files.
Class 'Cowpox' has 66 files.
Class 'Squamous cell carcinoma' has 628 files.
Class 'Measles' has 55 files.
Class 'Atopic Dermatitis' has 1257 files.
Class 'Vascular lesion' has 253 files.
Class 'hyperpigmentation' has 110 files.
Class 'Tinea Ringworm Candidiasis' has 2689 files.
Class 'Seborrheic Keratoses' has 4793 files.
Class 'Chicken Pox' has 75 files.
Class 'Eczema' has 2687 files.
Class 'Vasculitis' has 461 files.
Class 'Benign keratosis' has 3272 files.
Class 'Vascular_Tumors' has 543 files.
Class 'Candidiasis' has 248 files.
Class 'Warts Molluscum' has 2683 files.
Class 'Psoriasis Lichen Planus' has 3428 files.
Class 'Sunlight Damage' has 312 files.
Class 'Dermatofibroma' has 203 files.
Class 'Rosacea' has 254 files.
Class 'MonkeyPox' has 284 files.
Class 'Lupus' has 311 files.
Class 'Infesta

In [13]:
import random
import os
import cv2
import numpy as np

def random_rotation(image, angle_range=(-45, 45)):
    angle = random.uniform(angle_range[0], angle_range[1])
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h))
    return rotated

def horizontal_flip(image):
    return cv2.flip(image, 1)

def random_brightness(image, factor_range=(0.5, 1.5)):
    factor = random.uniform(factor_range[0], factor_range[1])
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 2] = hsv[:, :, 2] * factor
    hsv[:, :, 2][hsv[:, :, 2] > 255] = 255
    hsv = hsv.astype(np.uint8)
    bright_img = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    return bright_img

def random_zoom(image, zoom_range=(0.5, 1.5)):
    zoom_factor = random.uniform(zoom_range[0], zoom_range[1])
    (h, w) = image.shape[:2]
    new_h, new_w = int(h * zoom_factor), int(w * zoom_factor)
    # Resize the image with the zoom factor
    resized = cv2.resize(image, (new_w, new_h))
    
    if zoom_factor < 1:
        # Pad the image to the original size
        pad_h = (h - new_h) // 2
        pad_w = (w - new_w) // 2
        padded = cv2.copyMakeBorder(resized, pad_h, h - new_h - pad_h, pad_w, w - new_w - pad_w, cv2.BORDER_REFLECT_101)
        return padded
    else:
        # Crop the image to the original size
        start_h = (new_h - h) // 2
        start_w = (new_w - w) // 2
        cropped = resized[start_h:start_h+h, start_w:start_w+w]
        return cropped

# Optionally, you can create a function to combine multiple augmentations.
def aggressive_augmentation(image):
    # Apply rotation, then brightness, then zoom
    aug_img = random_rotation(image)
    aug_img = random_brightness(aug_img)
    aug_img = random_zoom(aug_img)
    # Optionally add a horizontal flip randomly as well
    if random.random() < 0.5:
        aug_img = horizontal_flip(aug_img)
    return aug_img

input_dir = '/kaggle/working/processed_images'
output_dir = '/kaggle/working/augmented_dataset'
os.makedirs(output_dir, exist_ok=True)

for class_name in os.listdir(input_dir):
    class_dir = os.path.join(input_dir, class_name)
    if not os.path.isdir(class_dir):
        continue
    output_class_dir = os.path.join(output_dir, class_name)
    os.makedirs(output_class_dir, exist_ok=True)
    image_files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    num_images = len(image_files)
    for file in image_files:
        img = cv2.imread(os.path.join(class_dir, file))
        if img is not None:
            cv2.imwrite(os.path.join(output_class_dir, file), img)
            
    if num_images < 500 and num_images > 100:
        needed = (5 * num_images) - num_images
        print(f"Augmenting class '{class_name}' with {needed} additional images.")
        for i in range(needed):
            file = random.choice(image_files)
            img = cv2.imread(os.path.join(class_dir, file))
            if img is None:
                continue
            # Use aggressive augmentation combining multiple transformations
            aug_img = aggressive_augmentation(img)
            new_filename = f"aug_{i}_{file}"
            cv2.imwrite(os.path.join(output_class_dir, new_filename), aug_img)
        print("Done")
        
    if num_images < 100:
        needed = 500 - num_images
        print(f"Augmenting class '{class_name}' with {needed} additional images.")
        for i in range(needed):
            file = random.choice(image_files)
            img = cv2.imread(os.path.join(class_dir, file))
            if img is None:
                continue
            aug_img = aggressive_augmentation(img)
            new_filename = f"aug_{i}_{file}"
            cv2.imwrite(os.path.join(output_class_dir, new_filename), aug_img)
        print("Done")

print("DONE!!")

Augmenting class 'Cowpox' with 434 additional images.
Done
Augmenting class 'Measles' with 445 additional images.
Done
Augmenting class 'Vascular lesion' with 1012 additional images.
Done
Augmenting class 'hyperpigmentation' with 440 additional images.
Done
Augmenting class 'Chicken Pox' with 425 additional images.
Done
Augmenting class 'Vasculitis' with 1844 additional images.
Done
Augmenting class 'Candidiasis' with 992 additional images.
Done
Augmenting class 'Sunlight Damage' with 1248 additional images.
Done
Augmenting class 'Dermatofibroma' with 812 additional images.
Done
Augmenting class 'Rosacea' with 1016 additional images.
Done
Augmenting class 'MonkeyPox' with 1136 additional images.
Done
Augmenting class 'Lupus' with 1244 additional images.
Done
DONE!!


In [14]:
import os


dataset_dir = '/kaggle/working/augmented_dataset'

avg = []
for class_name in os.listdir(dataset_dir):
    class_path = os.path.join(dataset_dir, class_name)
    if os.path.isdir(class_path):
        file_count = len([f for f in os.listdir(class_path) 
                          if os.path.isfile(os.path.join(class_path, f))])
        avg.append(file_count)
        print(f"Class '{class_name}' has {file_count} files.")


Class 'Basal Cell Carcinoma' has 5266 files.
Class 'SkinCancer' has 693 files.
Class 'Melanoma' has 4805 files.
Class 'DrugEruption' has 547 files.
Class 'Actinic keratosis' has 1600 files.
Class 'Cowpox' has 500 files.
Class 'Squamous cell carcinoma' has 628 files.
Class 'Measles' has 500 files.
Class 'Atopic Dermatitis' has 1257 files.
Class 'Vascular lesion' has 1265 files.
Class 'hyperpigmentation' has 550 files.
Class 'Tinea Ringworm Candidiasis' has 2689 files.
Class 'Seborrheic Keratoses' has 4793 files.
Class 'Chicken Pox' has 500 files.
Class 'Eczema' has 2687 files.
Class 'Vasculitis' has 2305 files.
Class 'Benign keratosis' has 3272 files.
Class 'Vascular_Tumors' has 543 files.
Class 'Candidiasis' has 1240 files.
Class 'Warts Molluscum' has 2683 files.
Class 'Psoriasis Lichen Planus' has 3428 files.
Class 'Sunlight Damage' has 1560 files.
Class 'Dermatofibroma' has 1015 files.
Class 'Rosacea' has 1270 files.
Class 'MonkeyPox' has 1420 files.
Class 'Lupus' has 1555 files.
Cla

In [10]:
sum(avg)/len(avg)

3101.0333333333333

In [ ]:
"""
import os
import shutil
output_directory = '/kaggle/working/not_required_images'
os.makedirs(output_directory, exist_ok = True)
print('OK')

Dataset_direct = '/kaggle/working/augmented_dataset'
destination_dir = '/kaggle/working/not_required_images'
for class_name in os.listdir(Dataset_direct):
    class_path = os.path.join(Dataset_direct, class_name)
    if os.path.isdir(class_path):
        file_count = len([f for f in os.listdir(class_path) 
        if os.path.isfile(os.path.join(class_path, f))])
        if file_count > 1500:
            all_images = os.listdir(class_path)
            num_to_choose = file_count - 1500
            selected_images = random.sample(all_images, num_to_choose)
            print(f"Selected {len(selected_images)} images from {class_path}")
            for img in selected_images:
                src = os.path.join(class_path, img)
                dst = os.path.join(destination_dir, img)
                shutil.move(src, dst)  # Use shutil.move() if you want to move instead of copy 
    print('Class Over')



print('DONE')
"""

In [ ]:
import os


dataset_dir = '/kaggle/working/augmented_dataset'

avg = []
for class_name in os.listdir(dataset_dir):
    class_path = os.path.join(dataset_dir, class_name)
    if os.path.isdir(class_path):
        file_count = len([f for f in os.listdir(class_path) 
                          if os.path.isfile(os.path.join(class_path, f))])
        avg.append(file_count)
        print(f"Class '{class_name}' has {file_count} files.")


In [ ]:
"""
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Set parameters
IMG_SIZE = (224, 224)  # EfficientNetB0 default input size
BATCH_SIZE = 32
DATASET_DIR = '/kaggle/working/augmented_dataset'  # Update to your dataset path

# Load dataset with training/validation split (20% validation)
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

validation_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

#normalization_layer = tf.keras.layers.experimental.preprocessing.Rescaling(1./255)
#train_dataset = train_dataset.map(lambda x, y: (normalization_layer(x), y))
#validation_dataset = validation_dataset.map(lambda x, y: (normalization_layer(x), y))

# Build the EfficientNetB0 model
# Load the base model with pretrained ImageNet weights, excluding the top layer.
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,))
base_model.trainable = False  # Freeze the base model initially

# Create new input and output layers for your 30 classes.
inputs = Input(shape=IMG_SIZE + (3,))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
outputs = Dense(30, activation='softmax')(x)  # 30 output classes
model = Model(inputs, outputs)

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',  # Use sparse loss if labels are integers
              metrics=['accuracy'])

# Display the model summary
model.summary()

# Train the model
EPOCHS = 10  # Adjust epochs as needed
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS
)

#Optional: Unfreeze some layers for fine-tuning after initial training
base_model.trainable = True
model.compile(optimizer=Adam(learning_rate=1e-5),
               loss='sparse_categorical_crossentropy',
               metrics=['accuracy'])
fine_tune_epochs = 5  # Adjust as needed
total_epochs = EPOCHS + fine_tune_epochs
history_fine = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=total_epochs,
    initial_epoch=EPOCHS
)

"""

In [17]:
import os
import shutil
import random

# Paths
original_dataset_dir = '/kaggle/working/augmented_dataset'  # This is the original folder containing class folders
output_base_dir = 'split_dataset'  # Folder where split data will go

# Ratios
train_ratio = 0.7
val_ratio = 0.2
test_ratio = 0.1

# Set random seed for reproducibility (optional)
random.seed(42)

# Create train, val, test directories
for split in ['train', 'val', 'test']:
    split_dir = os.path.join(output_base_dir, split)
    os.makedirs(split_dir, exist_ok=True)

# Process each class folder
for class_name in os.listdir(original_dataset_dir):
    class_dir = os.path.join(original_dataset_dir, class_name)

    if not os.path.isdir(class_dir):
        continue

    # Create class subfolders in each split directory
    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(output_base_dir, split, class_name), exist_ok=True)

    # Get all images in the class folder and shuffle
    images = os.listdir(class_dir)
    random.shuffle(images)

    # Determine split sizes
    total_images = len(images)
    train_size = int(total_images * train_ratio)
    val_size = int(total_images * val_ratio)
    test_size = total_images - train_size - val_size  # Ensures 100% coverage

    # Split images
    train_images = images[:train_size]
    val_images = images[train_size:train_size + val_size]
    test_images = images[train_size + val_size:]

    # Helper function to copy images to split folder
    def copy_images(image_list, dest_dir):
        for img in image_list:
            src = os.path.join(class_dir, img)
            dst = os.path.join(dest_dir, img)
            shutil.copy(src, dst)

    # Copy images into respective folders
    copy_images(train_images, os.path.join(output_base_dir, 'train', class_name))
    copy_images(val_images, os.path.join(output_base_dir, 'val', class_name))
    copy_images(test_images, os.path.join(output_base_dir, 'test', class_name))

    # Summary log
    print(f"Class {class_name}: {len(train_images)} train, {len(val_images)} val, {len(test_images)} test")

print(" Dataset split into train/val/test completed!")

Class Basal Cell Carcinoma: 3686 train, 1053 val, 527 test
Class SkinCancer: 485 train, 138 val, 70 test
Class Melanoma: 3363 train, 961 val, 481 test
Class DrugEruption: 382 train, 109 val, 56 test
Class Actinic keratosis: 1120 train, 320 val, 160 test
Class Cowpox: 350 train, 100 val, 50 test
Class Squamous cell carcinoma: 439 train, 125 val, 64 test
Class Measles: 350 train, 100 val, 50 test
Class Atopic Dermatitis: 879 train, 251 val, 127 test
Class Vascular lesion: 885 train, 253 val, 127 test
Class hyperpigmentation: 385 train, 110 val, 55 test
Class Tinea Ringworm Candidiasis: 1882 train, 537 val, 270 test
Class Seborrheic Keratoses: 3355 train, 958 val, 480 test
Class Chicken Pox: 350 train, 100 val, 50 test
Class Eczema: 1880 train, 537 val, 270 test
Class Vasculitis: 1613 train, 461 val, 231 test
Class Benign keratosis: 2290 train, 654 val, 328 test
Class Vascular_Tumors: 380 train, 108 val, 55 test
Class Candidiasis: 868 train, 248 val, 124 test
Class Warts Molluscum: 1878 t

In [ ]:
"""
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB5
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

strategy = tf.distribute.MirroredStrategy()
print("Number of devices: {}".format(strategy.num_replicas_in_sync))

with strategy.scope():
    train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "/kaggle/working/split_dataset/train",
    image_size=(224, 224),  # Adjust based on your model's input size
    batch_size=32,
    label_mode='int')  # or 'categorical' depending on your loss function

    val_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "/kaggle/working/split_dataset/val",
    image_size=(224, 224),
    batch_size=32,
    label_mode='int')

    # Define EfficientNetB5 model
    
    base_model = EfficientNetB5(weights='imagenet', include_top=False)

    # Freeze base model layers (optional, useful for transfer learning)
    base_model.trainable = False  

    # Add custom layers on top
    num_classes = 30
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dropout(0.4)(x)  # Regularization to prevent overfitting
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.2)(x)
    output_layer = Dense(num_classes, activation='softmax')(x)  # Adjust num_classes

    # Create final model
    model = Model(inputs=base_model.input, outputs=output_layer)

    # Compile model
    model.compile(optimizer=Adam(learning_rate=0.0001), 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

# Train model
epochs = 25
history = model.fit(train_dataset, validation_data=val_dataset, epochs=epochs)

# Unfreeze the base model for fine-tuning (optional)
#base_model.trainable = True
#model.compile(optimizer=Adam(learning_rate=0.00001), 
#             loss='sparse_categorical_crossentropy', 
#             metrics=['accuracy'])

# Fine-tune the model
#fine_tune_epochs = 25
#history_fine = model.fit(train_dataset, validation_data=val_dataset, epochs=fine_tune_epochs)

# Save model
"""

In [ ]:
"""
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB5
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "/kaggle/working/split_dataset/train",
    image_size=(224, 224),  # Adjust based on your model's input size
    batch_size=32,
    label_mode= 'int')  # or 'categorical' depending on your loss function

val_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "/kaggle/working/split_dataset/val",
    image_size=(224, 224),
    batch_size=32,
    label_mode='int')

    # Define EfficientNetB5 model
    
base_model = EfficientNetB5(weights='imagenet', include_top=False)

    # Freeze base model layers (optional, useful for transfer learning)
base_model.trainable = True  

    # Add custom layers on top
num_classes = 30
x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.4)(x)  # Regularization to prevent overfitting
x = Dense(256, activation='relu')(x)
x = Dropout(0.2)(x)
output_layer = Dense(num_classes, activation='softmax')(x)  # Adjust num_classes

    # Create final model
model = Model(inputs=base_model.input, outputs=output_layer)

    # Compile model
model.compile(optimizer=Adam(learning_rate=0.0001), 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

# Train model
epochs = 25
history = model.fit(train_dataset, validation_data=val_dataset, epochs=epochs)
"""

In [ ]:
#actual one
"""
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB5
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

strategy = tf.distribute.MirroredStrategy()
print("Number of devices: {}".format(strategy.num_replicas_in_sync))

with strategy.scope():
    train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
        "/kaggle/working/split_dataset/train",
        image_size=(224, 224),  # Adjust based on your model's input size
        batch_size=32,
        label_mode='int'  # or 'categorical' depending on your loss function
    )

    val_dataset = tf.keras.preprocessing.image_dataset_from_directory(
        "/kaggle/working/split_dataset/val",
        image_size=(224, 224),
        batch_size=32,
        label_mode='int'
    )

    # Define EfficientNetB5 model
    base_model = EfficientNetB5(weights='imagenet', include_top=False)
    base_model.trainable = True  # You can choose to freeze part of the model if needed

    # Build custom head
    num_classes = 30
    # Global Average Pooling over the convolutional feature maps
    x = GlobalAveragePooling2D()(base_model.output)

    # Batch normalization helps stabilize training and improve generalization
    x = BatchNormalization()(x)

    # Dropout to reduce overfitting
    x = Dropout(0.4)(x)

    # Dense layer with L2 regularization
    x = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(x)
    
    # Optional: Add another Batch Normalization layer after the Dense layer
    x = BatchNormalization()(x)
    
    # Another dropout layer
    x = Dropout(0.2)(x)

    # Final output layer with softmax activation
    output_layer = Dense(num_classes, activation='softmax')(x)

    # Create final model
    model = Model(inputs=base_model.input, outputs=output_layer)

    # Compile the model
    model.compile(optimizer=Adam(learning_rate=0.0001), 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])

# Train model
epochs = 25
history = model.fit(train_dataset, validation_data=val_dataset, epochs=epochs)

"""

In [19]:
import tensorflow as tf
import numpy as np
import os
from tensorflow.keras.applications import EfficientNetB5
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils import class_weight

# Distributed training strategy (optional)
strategy = tf.distribute.MirroredStrategy()
print("Number of devices: {}".format(strategy.num_replicas_in_sync))

# Directories and parameters
train_dir = "/kaggle/working/split_dataset/train"
val_dir = "/kaggle/working/split_dataset/val"
img_size = (224, 224)
batch_size = 32

# Load training dataset (without augmentation)
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    image_size=img_size,
    batch_size=batch_size,
    label_mode='int'
)

# Load validation dataset
val_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    val_dir,
    image_size=img_size,
    batch_size=batch_size,
    label_mode='int'
)

# Compute class weights to mitigate class imbalance
train_labels = []
for _, labels in train_dataset.unbatch():
    train_labels.append(labels.numpy())
train_labels = np.array(train_labels)

weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = dict(enumerate(weights))
print("Class weights:", class_weights)

with strategy.scope():
    # Build the EfficientNetB5 model
    base_model = EfficientNetB5(weights='imagenet', include_top=False,
                                 input_shape=(img_size[0], img_size[1], 3))
    base_model.trainable = True  # You can freeze layers if needed

    # Custom classification head
    x = GlobalAveragePooling2D()(base_model.output)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    x = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    
    # Determine number of classes from the subdirectories in train_dir
    num_classes = len(os.listdir(train_dir))
    output_layer = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=output_layer)

    # Compile the model
    model.compile(optimizer=Adam(learning_rate=0.0001), 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])

# Callbacks for early stopping and reducing learning rate
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
lr_reduce = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)

# Train the model
epochs = 25
history = model.fit(
    train_dataset, 
    validation_data=val_dataset, 
    epochs=epochs, 
    class_weight=class_weights,
    callbacks=[early_stop, lr_reduce]
)

print("Training complete!")


Number of devices: 2
Found 43291 files belonging to 30 classes.
Found 12364 files belonging to 30 classes.
Class weights: {0: 0.6586185912064506, 1: 1.2884226190476191, 2: 1.641676147136898, 3: 0.39149032374751314, 4: 0.6301455604075692, 5: 4.099526515151515, 6: 1.6624807987711214, 7: 4.122952380952381, 8: 4.122952380952381, 9: 2.032441314553991, 10: 3.777574171029668, 11: 0.7675709219858156, 12: 3.9427140255009108, 13: 1.3263174019607844, 14: 4.122952380952381, 15: 0.22579147759870652, 16: 0.42909108930518386, 17: 1.4532057737495805, 18: 0.6015145199388634, 19: 1.62320959880015, 20: 0.4301142573273721, 21: 2.9753264604811, 22: 3.2870918754745633, 23: 1.3214590964590964, 24: 0.7667552249380092, 25: 1.6305461393596987, 26: 3.797456140350877, 27: 0.8946269890473239, 28: 0.7683883564075258, 29: 3.748138528138528}
Epoch 1/25
1353/1353 ━━━━━━━━━━━━━━━━━━━━ 1112s 747ms/step - accuracy: 0.2264 - loss: 3.3757 - val_accuracy: 0.5149 - val_loss: 1.9704 - learning_rate: 1.0000e-04
Epoch 2/25
1353

In [20]:
model.save("new_efficientnetb5_model.h5")
print('DONE')

DONE


In [21]:
from tensorflow.keras.models import load_model

model = load_model('/kaggle/working/new_efficientnetb5_model.h5')
test_dir = '/kaggle/working/split_dataset/test'

test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),  # adjust based on your model's input
    batch_size=32,
    shuffle=False  # No need to shuffle for evaluation
)

test_loss, test_acc = model.evaluate(test_dataset)  # test_dataset should be a tf.data.Dataset or similar
print('Test accuracy:', test_acc)


Found 6204 files belonging to 30 classes.
194/194 ━━━━━━━━━━━━━━━━━━━━ 63s 217ms/step - accuracy: 0.7876 - loss: 0.9826
Test accuracy: 0.7944874167442322


In [25]:
import cv2
import numpy as np
import os

def predict_disease(model, image_path, class_names):
    # Read the image using OpenCV
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError("Image not found or unable to read.")

    # Convert BGR (OpenCV default) to RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Resize image to 224x224 (as used during training)
    img_resized = cv2.resize(img_rgb, (224, 224))
    
    # If any preprocessing was applied during training (e.g., normalization),
    # apply it here as well. In our pipeline, images were loaded as-is.
    # For example, if you need to scale pixel values between 0 and 1, do:
    # img_preprocessed = img_resized / 255.0
    # For now, we'll keep the pixel values in [0,255]:
    img_preprocessed = np.array(img_resized, dtype=np.float32)
    
    # Add batch dimension (model expects input shape of (batch_size, 224, 224, 3))
    img_batch = np.expand_dims(img_preprocessed, axis=0)
    
    # Get model prediction (output is a probability distribution)
    predictions = model.predict(img_batch)
    
    # Get index of the highest probability
    predicted_index = np.argmax(predictions, axis=1)[0]
    
    # Map the index to the corresponding disease/class name
    predicted_class = class_names[predicted_index]
    return predicted_class

# Example usage:

# Suppose your training data was split into folders in '/kaggle/working/split_dataset/train'
train_dir = '/kaggle/working/split_dataset/train'
# The class names are typically sorted alphabetically:
class_names = sorted(os.listdir(train_dir))

# Replace 'path/to/your/image.jpg' with the path to your image file
image_path = '/kaggle/input/testing/chikenpox.jpg'
predicted_disease = predict_disease(model, image_path, class_names)
print("Predicted Disease:", predicted_disease)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
Predicted Disease: Actinic keratosis


In [ ]:
import cv2
image = cv2.imread('/kaggle/working/augmented_dataset/Chicken Pox/aug_123_CHP_04_05.jpg')
cv2.imshow('my image',image)